# Exploratory Data Analysis

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
processed_dir = Path("../data/processed")
reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
fm = pd.read_csv(processed_dir / "clean_fund_master.csv")
nav = pd.read_csv(processed_dir / "clean_nav.csv")
aum = pd.read_csv(processed_dir / "clean_aum_by_fund_house.csv")
sip = pd.read_csv(processed_dir / "clean_monthly_sip_inflows.csv")
cat = pd.read_csv(processed_dir / "clean_category_inflows.csv")
folios = pd.read_csv(processed_dir / "clean_industry_folio_count.csv")
perf = pd.read_csv(processed_dir / "clean_performance.csv")
tx = pd.read_csv(processed_dir / "clean_transactions.csv")
holdings = pd.read_csv(processed_dir / "clean_portfolio_holdings.csv")
bench = pd.read_csv(processed_dir / "clean_benchmark_indices.csv")

nav['date'] = pd.to_datetime(nav['date'])
sip['month'] = pd.to_datetime(sip['month'] + "-01")
folios['month'] = pd.to_datetime(folios['month'] + "-01")
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'])
holdings['portfolio_date'] = pd.to_datetime(holdings['portfolio_date'])
bench['date'] = pd.to_datetime(bench['date'])

### 1. NAV Trends

In [ ]:
selected = [119551, 120503, 118632, 119092, 120841]
df_subset = nav[nav['amfi_code'].isin(selected)].merge(fm[['amfi_code', 'scheme_name']], on='amfi_code')

fig1 = px.line(df_subset, x='date', y='nav', color='scheme_name', title='Daily NAV Trend')
fig1.add_vrect(x0="2023-01-01", x1="2023-12-31", fillcolor="green", opacity=0.1, line_width=0, annotation_text="2023 Bull Run")
fig1.add_vrect(x0="2024-03-01", x1="2024-06-01", fillcolor="red", opacity=0.1, line_width=0, annotation_text="2024 Corrections")

fig1.write_image("../reports/nav_trends.png")
fig1.show()

### 2. AUM Growth

In [ ]:
aum['year'] = pd.to_datetime(aum['date']).dt.year
yearly_aum = aum.groupby(['year', 'fund_house'])['aum_crore'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=yearly_aum, x='year', y='aum_crore', hue='fund_house')
plt.title("Yearly AUM Growth by Fund House (2022-2025)")
plt.ylabel("AUM (Crores)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("../reports/aum_growth.png")
plt.show()

### 3. SIP Inflows

In [ ]:
fig3 = px.line(sip, x='month', y='sip_inflow_crore', title='Monthly SIP Inflows')
fig3.add_annotation(x="2025-12-01", y=31002, text="Peak (₹31,002 Cr)", showarrow=True, arrowhead=1)
fig3.write_image("../reports/sip_inflow_trends.png")
fig3.show()

### 4. Category Inflows Heatmap

In [ ]:
pivot_df = cat.pivot(index='category', columns='month', values='net_inflow_crore')
plt.figure(figsize=(12, 6))
sns.heatmap(pivot_df, cmap='YlGnBu', annot=True, fmt=".0f")
plt.title("Monthly Net Category Inflows")
plt.tight_layout()
plt.savefig("../reports/category_inflows_heatmap.png")
plt.show()

### 5. Demographics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

age_counts = tx['age_group'].value_counts()
axes[0].pie(age_counts, labels=age_counts.index, autopct='%1.1f%%', startangle=140)
axes[0].set_title("Age Distribution")

sns.boxplot(ax=axes[1], data=tx[tx['transaction_type'] == 'SIP'], x='age_group', y='amount_inr', hue='gender', palette='Set2')
axes[1].set_title("SIP Amount by Age & Gender")

plt.tight_layout()
plt.savefig("../reports/investor_demographics.png")
plt.show()

### 6. Geographic Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

state_sip = tx[tx['transaction_type'] == 'SIP'].groupby('state')['amount_inr'].sum().sort_values(ascending=False).head(10)
sns.barplot(ax=axes[0], x=state_sip.values, y=state_sip.index, palette='viridis')
axes[0].set_title("Top 10 States by SIP amount")

tier_counts = tx['city_tier'].value_counts()
axes[1].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title("City Tier Contribution")

plt.tight_layout()
plt.savefig("../reports/geographic_distribution.png")
plt.show()

### 7. Folio Growth

In [ ]:
fig7 = px.line(folios, x='month', y='total_folios_crore', title='Folio Growth')
fig7.add_annotation(x="2022-01-01", y=13.26, text="13.26 Cr", showarrow=True)
fig7.add_annotation(x="2025-12-01", y=26.12, text="26.12 Cr", showarrow=True)
fig7.write_image("../reports/folio_growth.png")
fig7.show()

### 8. Daily Return Correlation

In [ ]:
subset_codes = fm['amfi_code'].head(10).tolist()
returns_subset = nav[nav['amfi_code'].isin(subset_codes)].pivot(index='date', columns='amfi_code', values='daily_return_pct')

name_map = fm.set_index('amfi_code')['scheme_name'].to_dict()
returns_subset = returns_subset.rename(columns=name_map)
corr = returns_subset.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Matrix (10 Selected Funds)")
plt.tight_layout()
plt.savefig("../reports/nav_correlation_matrix.png")
plt.show()

### 9. Sector Weightings

In [ ]:
sector_w = holdings.groupby('sector')['weight_pct'].sum().sort_values(ascending=False).head(8)
other_w = holdings.groupby('sector')['weight_pct'].sum().sort_values(ascending=False)[8:].sum()
sector_w['Others'] = other_w

plt.figure(figsize=(8, 8))
plt.pie(sector_w.values, labels=sector_w.index, autopct='%1.1f%%', startangle=90, pctdistance=0.85)
circle = plt.Circle((0,0), 0.70, fc='white')
plt.gcf().gca().add_artist(circle)

plt.title("Sector Weight Allocation")
plt.tight_layout()
plt.savefig("../reports/sector_allocation_donut.png")
plt.show()

### 10. Summary Findings

1. **NAV Rallies**: Key large-cap and debt funds experienced a strong bull run throughout 2023, followed by short corrections in early-to-mid 2024 (Reference: *NAV Trend Analysis*).
2. **AUM Concentration**: SBI Mutual Fund maintains a dominant position, leading the industry AUM with a size of over ₹12.5L Cr (Reference: *Yearly AUM Growth*).
3. **SIP Expansion**: Monthly SIP inflows showed a continuous upward trend, peaking at an all-time high of ₹31,002 Cr in December 2025 (Reference: *Monthly SIP Inflow Trend*).
4. **Active Category Inflows**: Large Cap and Mid Cap categories remain the largest contributors to net category inflows throughout the year (Reference: *Monthly Category Inflows*).
5. **Young Investor Growth**: The 26-35 age group represents the largest cohort of investors (over 40%), signaling strong retail adoption by young professionals (Reference: *Investor Age Group Distribution*).
6. **Demographic Allocation**: Across all age brackets, male investors show slightly higher average box plot SIP sizes than female investors, though transaction counts remain strong across both genders (Reference: *SIP Transaction Amount by Age & Gender*).
7. **Regional Leaders**: Punjab, Tamil Nadu, and Madhya Pradesh are the top states by total SIP investment amounts (Reference: *Top 10 States by Total SIP Investment*).
8. **Tier-30 Domination**: Metro and Tier-30 cities contribute over 60% of the transaction volume compared to Beyond-30 cities (Reference: *T30 vs B30 City Tier Contribution*).
9. **Folio Growth**: Total mutual fund folios doubled in 4 years, growing from 13.26 Cr to 26.12 Cr (Reference: *Mutual Fund Folio Growth Trend*).
10. **Sector Bias**: Banking and Financial Services represent the highest average sector weight allocation (~14%) across all equity portfolios (Reference: *Sector Weight Allocation*).